In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('Nike Dataset.csv')

# Drop 'Total Sales' column as it causes leakage
df = df.drop(columns=['Total Sales'])

# Convert 'Invoice Date' to datetime
df['Invoice Date'] = pd.to_datetime(df['Invoice Date'], format='%d-%m-%Y')

# Feature Engineering: Add time-based features
df['Year'] = df['Invoice Date'].dt.year
df['Month'] = df['Invoice Date'].dt.month
df['Week'] = df['Invoice Date'].dt.isocalendar().week  # Use isocalendar() to get the week number
df['Day'] = df['Invoice Date'].dt.dayofweek

df.head()

,Invoice Date,Product,Region,Retailer,Sales Method,State,Price per Unit,Units Sold,Year,Month,Week,Day
0,2020-01-01,Men's Street Footwear,Northeast,Foot Locker,In-store,New York,50,120,2020,1,1,2
1,2020-01-02,Men's Athletic Footwear,Northeast,Foot Locker,In-store,New York,50,100,2020,1,1,3
2,2020-01-03,Women's Street Footwear,Northeast,Foot Locker,In-store,New York,40,100,2020,1,1,4
3,2020-01-04,Women's Athletic Footwear,Northeast,Foot Locker,In-store,New York,45,85,2020,1,1,5
4,2020-01-05,Men's Apparel,Northeast,Foot Locker,In-store,New York,60,90,2020,1,1,6


In [2]:
# Generate lag features for sales prediction (Lag1, Lag2, Lag3, etc.)
df['Sales Lag1'] = df['Units Sold'].shift(1)
df['Sales Lag2'] = df['Units Sold'].shift(2)
df['Sales Lag3'] = df['Units Sold'].shift(3)
df['Sales Lag4'] = df['Units Sold'].shift(4)

# Drop rows with missing values (first few rows will have NaN for lag values)
df = df.dropna()

# Features and Target
X = df[['Sales Lag1', 'Sales Lag2', 'Sales Lag3', 'Sales Lag4', 'Year', 'Month', 'Week', 'Day']]
y = df['Units Sold']

# Train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [4]:
import xgboost as xgb  # Import xgboost
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Train XGBoost model
xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42)
xgb_model.fit(X_train, y_train)

# Predict on the test set using XGBoost
xgb_y_pred = xgb_model.predict(X_test)

# Calculate RMSE for XGBoost
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_y_pred))
# Calculate R² for XGBoost
xgb_r2 = r2_score(y_test, xgb_y_pred)

print(f"XGBoost RMSE: {xgb_rmse}")
print(f"XGBoost R²: {xgb_r2}")

# Train Random Forest with more trees
rf_model = RandomForestRegressor(n_estimators=500, random_state=42)  # Increased number of trees
rf_model.fit(X_train, y_train)

# Predict on the test set using Random Forest
rf_y_pred = rf_model.predict(X_test)

# Calculate RMSE for Random Forest
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_y_pred))
# Calculate R² for Random Forest
rf_r2 = r2_score(y_test, rf_y_pred)

print(f"Random Forest RMSE: {rf_rmse}")
print(f"Random Forest R²: {rf_r2}")

# Compare RMSE and R² values and choose the best model
if xgb_rmse < rf_rmse:
    print("XGBoost performs better in terms of RMSE, choosing XGBoost for prediction.")
else:
    print("Random Forest performs better in terms of RMSE, choosing Random Forest for prediction.")

if xgb_r2 > rf_r2:
    print("XGBoost performs better in terms of R², choosing XGBoost for prediction.")
else:
    print("Random Forest performs better in terms of R², choosing Random Forest for prediction.")


XGBoost RMSE: 11.010489231088242
XGBoost R²: 0.7659644484519958
Random Forest RMSE: 10.908952706432261
Random Forest R²: 0.7702609973754837
Random Forest performs better in terms of RMSE, choosing Random Forest for prediction.
Random Forest performs better in terms of R², choosing Random Forest for prediction.


In [5]:
from sklearn.model_selection import RandomizedSearchCV

# Define the hyperparameters grid to search over
param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2', None]
}

# Initialize Random Forest model
rf_model = RandomForestRegressor(random_state=42)

# Perform RandomizedSearchCV
random_search = RandomizedSearchCV(rf_model, param_distributions=param_dist, n_iter=10, cv=3, random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

# Best parameters from RandomizedSearchCV
print(f"Best parameters: {random_search.best_params_}")

# Use the best model to predict
rf_best_model = random_search.best_estimator_
rf_best_y_pred = rf_best_model.predict(X_test)

# Calculate RMSE and R² for the tuned Random Forest model
rf_best_rmse = np.sqrt(mean_squared_error(y_test, rf_best_y_pred))
rf_best_r2 = r2_score(y_test, rf_best_y_pred)

print(f"Tuned Random Forest RMSE: {rf_best_rmse}")
print(f"Tuned Random Forest R²: {rf_best_r2}")


Best parameters: {'n_estimators': 400, 'min_samples_split': 5, 'max_features': 'log2', 'max_depth': 20}
Tuned Random Forest RMSE: 10.641668570521526
Tuned Random Forest R²: 0.7813809155395086


In [9]:
# Evaluate Random Forest model on the test set
rf_y_pred_test = rf_model.predict(X_test)  # Predict on the test set

# Calculate RMSE for Random Forest on the test set
rf_rmse_test = np.sqrt(mean_squared_error(y_test, rf_y_pred_test))

# Calculate R² for Random Forest on the test set
rf_r2_test = r2_score(y_test, rf_y_pred_test)

# Print the results
print(f"Random Forest Test RMSE: {rf_rmse_test}")
print(f"Random Forest Test R²: {rf_r2_test}")

# Evaluate XGBoost model on the test set
xgb_y_pred_test = xgb_model.predict(X_test)  # Predict on the test set

# Calculate RMSE for XGBoost on the test set
xgb_rmse_test = np.sqrt(mean_squared_error(y_test, xgb_y_pred_test))

# Calculate R² for XGBoost on the test set
xgb_r2_test = r2_score(y_test, xgb_y_pred_test)

# Print the results
print(f"XGBoost Test RMSE: {xgb_rmse_test}")
print(f"XGBoost Test R²: {xgb_r2_test}")

# Compare RMSE and R² values for both models on the test set
if xgb_rmse_test < rf_rmse_test:
    print("XGBoost performs better on the test set in terms of RMSE, choosing XGBoost for prediction.")
else:
    print("Random Forest performs better on the test set in terms of RMSE, choosing Random Forest for prediction.")

if xgb_r2_test > rf_r2_test:
    print("XGBoost performs better on the test set in terms of R², choosing XGBoost for prediction.")
else:
    print("Random Forest performs better on the test set in terms of R², choosing Random Forest for prediction.")


Random Forest Test RMSE: 9.826789002752422
Random Forest Test R²: 0.7621910257493744
XGBoost Test RMSE: 10.762832434434662
XGBoost Test R²: 0.7147286534309387
Random Forest performs better on the test set in terms of RMSE, choosing Random Forest for prediction.
Random Forest performs better on the test set in terms of R², choosing Random Forest for prediction.


In [8]:
import numpy as np
import pandas as pd

# Assuming the Random Forest model has been trained and tuned
# Let's assume the following columns are part of the dataset
# 'Sales Lag1', 'Sales Lag2', 'Sales Lag3', 'Sales Lag4', 'Year', 'Month', 'Week', 'Day'

# Example function to predict future sales based on the most recent data
def predict_future_sales(rf_model, latest_data, weeks_to_predict=4):
    future_sales = []
    
    # Predict the next 'weeks_to_predict' weeks
    for _ in range(weeks_to_predict):
        # Make prediction for the next week
        prediction = rf_model.predict(latest_data)
        future_sales.append(prediction[0])
        
        # Update the latest data to reflect the prediction for the next week
        # Shift the lag features for the new prediction
        latest_data['Sales Lag4'] = latest_data['Sales Lag3']
        latest_data['Sales Lag3'] = latest_data['Sales Lag2']
        latest_data['Sales Lag2'] = latest_data['Sales Lag1']
        latest_data['Sales Lag1'] = prediction[0]
    
    return future_sales

# Assuming the user provides the latest sales data as input
def get_user_input():
    # Example: User enters the sales data for the most recent week (last available data)
    print("Enter the following details for the most recent week:")
    sales_lag1 = float(input("Sales from last week (Sales Lag 1): "))
    sales_lag2 = float(input("Sales from 2 weeks ago (Sales Lag 2): "))
    sales_lag3 = float(input("Sales from 3 weeks ago (Sales Lag 3): "))
    sales_lag4 = float(input("Sales from 4 weeks ago (Sales Lag 4): "))
    year = int(input("Year: "))
    month = int(input("Month (1-12): "))
    week = int(input("Week number: "))
    day = int(input("Day of the week (0=Monday, 6=Sunday): "))
    
    # Return as a DataFrame
    latest_data = pd.DataFrame({
        'Sales Lag1': [sales_lag1],
        'Sales Lag2': [sales_lag2],
        'Sales Lag3': [sales_lag3],
        'Sales Lag4': [sales_lag4],
        'Year': [year],
        'Month': [month],
        'Week': [week],
        'Day': [day]
    })
    
    return latest_data

# Example usage: Get user input and predict future sales
latest_data = get_user_input()

# Predict future sales for the next 4 weeks
future_sales = predict_future_sales(rf_model, latest_data)

# Output the predictions
print("Predicted sales for the next 4 weeks:")
for i, sales in enumerate(future_sales, 1):
    print(f"Week {i}: {sales:.2f} units")



Enter the following details for the most recent week:


Sales from last week (Sales Lag 1):  500
Sales from 2 weeks ago (Sales Lag 2):  140
Sales from 3 weeks ago (Sales Lag 3):  120
Sales from 4 weeks ago (Sales Lag 4):  130
Year:  2024
Month (1-12):  6
Week number:  24
Day of the week (0=Monday, 6=Sunday):  3


Predicted sales for the next 4 weeks:
Week 1: 44.34 units
Week 2: 48.50 units
Week 3: 54.63 units
Week 4: 57.84 units


In [10]:
import joblib

# Save the trained Random Forest model
joblib.dump(rf_model, 'random_forest_model11.pkl')

# Optionally, save the XGBoost model if you want to use it too
joblib.dump(xgb_model, 'xgboost_model11.pkl')


['xgboost_model11.pkl']